# Lab 2 — Decoding (EP01)

ใน Lab 1 เรานับ token ที่ model อ่านและเขียน lab นี้เราจะดูว่า model **เลือก** token ถัดไปอย่างไร

**สิ่งที่จะได้เรียน**
1. ขอความน่าจะเป็นจริงของ token ถัดไปจาก Gemma (`logprobs`)
2. เขียน greedy, temperature, top-k, top-p และ min-p เองทีละฟังก์ชัน
3. สุ่ม token ด้วยโค้ดของเรา แล้วเทียบกับ `temperature` ของ API จริง

ความน่าจะเป็นและคำตอบของ model ทั้งหมดมาจากการรันจริง — ตัวเลขของคุณอาจต่างจากในคลิปเล็กน้อย

> อ้างอิง: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §1.12 — CC BY-SA 4.0

> ✏️ **Starter notebook — ฉบับให้ผู้เรียนลงมือเขียน**
>
> โค้ดส่วนใหญ่เตรียมไว้ให้แล้ว เขียนเพิ่มเฉพาะ **5 จุดที่มี `✏️ TODO`** (แทนที่ `___` หรือ `...` ด้วยโค้ดของคุณ) ทุกจุดบอกเวลาในวิดีโอที่อธิบายโค้ดตรงนั้น (▶ m:ss) — ดูคลิปก่อน แล้วลองเขียนเองโดยไม่เปิดเฉลย
>
> รันทุก cell ตามลำดับจากบนลงล่าง ถ้า cell ไหนยังเติมไม่ครบจะเจอ `NameError` หรือ `TypeError` เป็นเรื่องปกติ
>
> ติดตรงไหน เปิดเฉลยใน **`ep01-lab2-decoding.ipynb`** (โค้ดชุดเดียวกับในวิดีโอ)

## 0. เตรียมเครื่อง

ต้องมี Python 3.10+ (หรือ Colab) และ `OPENROUTER_API_KEY` — วิธีตั้งค่า key และเหตุผลที่เลือก model นี้อยู่ใน **Lab 1**

In [ ]:
%pip install -q "openai>=1.50" python-dotenv

In [ ]:
# ---- จาก Lab 1: ตั้งค่า client ----
import os, math, random
from collections import Counter
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"

In [ ]:
PROVIDER = {"order": ["google-vertex/global"], "allow_fallbacks": False}
EXTRA = {"reasoning": {"enabled": False}, "provider": PROVIDER}

## 1. ความน่าจะเป็นจริงของ token ถัดไป

ขอ `logprobs=True, top_logprobs=10` กับ `max_tokens=1` → API คืน 10 token ที่น่าจะเป็นที่สุดสำหรับตำแหน่งแรกของคำตอบ
พร้อม **logprob** (log ของความน่าจะเป็น) — `math.exp(logprob)` แปลงกลับเป็นความน่าจะเป็น 0–1

In [ ]:
def next_token_probs(prompt):
    resp = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}],
        max_tokens=1, logprobs=True, top_logprobs=10, extra_body=EXTRA,
    )
    top = resp.choices[0].logprobs.content[0].top_logprobs
    # ✏️ TODO 1 (▶ วิดีโอ 2:27): แปลง logprob ของแต่ละตัวเลือกเป็นความน่าจะเป็น 0–1
    probs = {t.token: ___ for t in top}
    return dict(sorted(probs.items(), key=lambda kv: -kv[1]))

In [ ]:
def show(probs, n=6):
    for tok, p in list(probs.items())[:n]:
        print(f"{tok!r:>14} {p:6.3f} {'█' * round(p * 40)}")

In [ ]:
PARIS = next_token_probs("Complete with one word only. The capital of France is")
show(PARIS)

In [ ]:
FOOD = next_token_probs("Complete with one word only. I enjoy eating")
show(FOOD)

## 2. Greedy และ temperature

**Greedy** = เลือกตัวที่สูงสุดทุกครั้ง · **Temperature** = ยกกำลังความน่าจะเป็นด้วย `1/T` แล้วทำให้รวมกันได้ 1
(ให้ผลเหมือนสูตรในตำรา) — `T < 1` ตัวสูงยิ่งสูง, `T > 1` ทุกตัวเข้าใกล้กัน

In [ ]:
def greedy(probs):
    return max(probs, key=probs.get)

print(greedy(PARIS), greedy(FOOD))

In [ ]:
def normalize(weights):
    total = sum(weights.values())
    return {t: w / total for t, w in weights.items()}

# ✏️ TODO 2 (▶ วิดีโอ 5:05): ยกกำลังความน่าจะเป็นแต่ละตัวด้วย 1/T

def temperature(probs, T):
    return normalize({t: ___ for t, p in probs.items()})

In [ ]:
for T in [0.5, 1, 2]:
    print(f"T = {T}")
    show(temperature(FOOD, T), n=4)

## 3. ตัดหาง: top-k, top-p, min-p

| วิธี | เก็บไว้ |
|---|---|
| top-k | `k` ตัวแรกเสมอ |
| top-p | เรียงจากมากไปน้อย เก็บจนผลรวมถึง `p` |
| min-p | ทุกตัวที่ได้อย่างน้อย `m ×` ของตัวที่หนึ่ง |

In [ ]:
# ✏️ TODO 3 (▶ วิดีโอ 6:16): เก็บ k ตัวแรกของ probs (เรียงจากมากไปน้อยแล้ว)
def top_k(probs, k):
    return normalize(dict(___))

def top_p(probs, p):
    keep, total = {}, 0
    for t, x in probs.items():
        keep[t] = x
        total += x
        # ✏️ TODO 4 (▶ วิดีโอ 6:34): หยุดเมื่อผลรวมความน่าจะเป็นถึง p
        if ___:
            break
    return normalize(keep)

# ✏️ TODO 5 (▶ วิดีโอ 6:44): เก็บเฉพาะตัวที่ได้อย่างน้อย m เท่าของตัวที่หนึ่ง

def min_p(probs, m):
    top = max(probs.values())
    return normalize({t: x for t, x in probs.items() if ___})

In [ ]:
for name, probs in [("Paris", PARIS), ("eating", FOOD)]:
    print(name)
    print("  top-k=5   :", list(top_k(probs, 5)))
    print("  top-p=0.95:", list(top_p(probs, 0.95)))
    print("  min-p=0.05:", list(min_p(probs, 0.05)))

## 4. สุ่มจริง

`random.choices` สุ่มตามน้ำหนัก · `random.seed(0)` ทำให้การสุ่มของเราได้ผลเดิมทุกครั้งที่รัน

In [ ]:
def sample(probs):
    return random.choices(list(probs), weights=list(probs.values()))[0]

random.seed(0)
for T in [0.3, 1.5]:
    picks = [sample(top_p(temperature(FOOD, T), 0.95)) for _ in range(20)]
    print(f"T = {T}:", Counter(picks))

## 5. เทียบกับ `temperature` ของ API

ถาม Gemma คำถามเดิม 10 ครั้งที่ `temperature=0` และ `1.5` ให้ server สุ่มเอง
(ทดสอบ 2026-11-09: `seed` เดียวกันบน server ยังได้คำตอบต่างกันได้ — seed ของ API ช่วยให้ผล "ใกล้เดิม" ไม่ได้รับประกันว่า "เหมือนเดิม")

In [ ]:
for T in [0, 1.5]:
    answers = []
    for _ in range(10):
        r = client.chat.completions.create(
            model=MODEL, messages=[{"role": "user", "content": "Complete with one word only. I enjoy eating"}],
            max_tokens=3, temperature=T, extra_body=EXTRA,
        )
        answers.append(r.choices[0].message.content.strip())
    print(f"T = {T}:", Counter(answers))

## สรุป

| เราได้เห็น | จาก lab นี้ |
|---|---|
| model ให้ความน่าจะเป็นกับทุกตัวเลือก | Paris ≈ 1.0 · pizza ≈ 0.8 ตามด้วย food, everything |
| greedy เลือกตัวสูงสุดเสมอ | Paris, pizza ทุกครั้ง |
| temperature เปลี่ยนรูปร่าง | T = 0.5 pizza เกือบเต็ม · T = 2 เหลือราวครึ่ง |
| top-k จำนวนตายตัว · top-p / min-p ปรับตามความมั่นใจ | Paris: 5 vs 1 ตัว · eating: 3 ตัว |
| API `temperature=0` ≈ greedy | pizza 10/10 |

**แบบฝึกหัด:** เขียน prompt ของคุณเองหนึ่งข้อที่คิดว่าคำตอบกระจาย แล้วดูว่า `top_p(…, 0.95)` เก็บไว้กี่ตัว
ลองเปลี่ยนเป็น T = 2 ก่อนตัด แล้วนับใหม่

▶ **ถัดไป: Lab 3 — ชื่อ model, provider, quantization และ modality** ทำไม Gemma ตัวเดียวกันแต่ต่างเจ้าจึงให้ผลไม่เท่ากัน